# C3-LDM Model Evaluation

This notebook evaluates a trained C3-LDM model on the test set and visualizes the results.

**Features:**
- Load trained checkpoint
- Evaluate on stratified test split
- Compute comprehensive metrics (RMSE, MAE, R², correlation, etc.)
- Visualize predictions vs ground truth
- Error analysis by population density
- Save results to files

## Setup

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.0.dev20250729+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5080


In [17]:
# Add parent directory to path for imports
import sys
sys.path.insert(0, os.path.abspath('.'))

from data.dataset import MultiProductDataset
from models import (
    BaselineDasymetric,
    ResidualVAE,
    TimeEmbedding,
    DualBranchConditionalEncoder,
    ProductEmbedding,
    SimpleUNet,
    C3LDMSampler
)
from eval.metrics import batch_metrics, print_metrics

## Configuration

In [20]:
# Configuration
CONFIG = {
    # Model checkpoint
    'checkpoint_path': 'checkpoints_wp/checkpoint_latest.pt',
    
    # Data
    'data_root': 'data',
    'test_csv': 'data/paired_dataset/test_split.csv',
    'products': ['WorldPop'],
    
    # Evaluation settings
    'num_samples': 1,  # Number of samples to generate per input (1=deterministic)
    'sampler': 'ddim',  # 'ddpm' or 'ddim'
    'num_steps': 50,    # Number of DDIM steps (ignored for DDPM)
    'max_eval_samples': None,  # None = evaluate all, or set to int (e.g., 100)
    
    # Output
    'output_dir': 'eval_results',
    'save_predictions': True,
    'num_visualize': 10,  # Number of samples to visualize
    
    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create output directory
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  checkpoint_path: checkpoints_wp/checkpoint_latest.pt
  data_root: data
  test_csv: data/paired_dataset/test_split.csv
  products: ['WorldPop']
  num_samples: 1
  sampler: ddim
  num_steps: 50
  max_eval_samples: None
  output_dir: eval_results
  save_predictions: True
  num_visualize: 10
  device: cuda


## Load Model

In [21]:
def load_models(checkpoint_path, device):
    """Load trained models from checkpoint."""
    print(f"Loading models from: {checkpoint_path}")
    
    # Initialize models
    models = {
        'baseline': BaselineDasymetric().to(device),
        'vae': ResidualVAE(latent_channels=4, base_channels=64).to(device),
        'time_emb': TimeEmbedding(dim=256, base_dim=64).to(device),
        'cond_encoder': DualBranchConditionalEncoder(cond_channels=256, low_res_ch=128, high_res_ch=128).to(device),
        'product_emb': ProductEmbedding(num_products=3, d_prod=64, cond_channels=256).to(device),
        'unet': SimpleUNet(in_channels=4, model_channels=128, time_emb_dim=256, cond_channels=256).to(device)
    }
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load learned model states
    learned_models = ['vae', 'time_emb', 'cond_encoder', 'product_emb', 'unet']
    for name in learned_models:
        if name in checkpoint['models']:
            models[name].load_state_dict(checkpoint['models'][name])
            print(f"  ✓ Loaded {name}")
        else:
            print(f"  ✗ Warning: {name} not found in checkpoint")
    
    # Set all models to eval mode
    for model in models.values():
        model.eval()
    
    # Get diffusion schedule
    if 'config' in checkpoint:
        config = checkpoint['config']
        T = getattr(config, 'diffusion_steps', 1000)
        beta_start = getattr(config, 'beta_start', 0.0001)
        beta_end = getattr(config, 'beta_end', 0.02)
    else:
        T, beta_start, beta_end = 1000, 0.0001, 0.02
    
    betas = torch.linspace(beta_start, beta_end, T).to(device)
    
    print(f"\nDiffusion schedule: T={T}, β=[{beta_start:.4f}, {beta_end:.4f}]")
    
    return models, betas, checkpoint

# Load models
models, betas, checkpoint = load_models(CONFIG['checkpoint_path'], CONFIG['device'])

# Print checkpoint info
if 'epoch' in checkpoint:
    print(f"\nCheckpoint info:")
    print(f"  Epoch: {checkpoint['epoch']}")
    print(f"  Step: {checkpoint.get('step', 'N/A')}")
    if 'metrics' in checkpoint:
        print(f"  Training loss: {checkpoint['metrics'].get('loss', 'N/A'):.4f}")

Loading models from: checkpoints_wp/checkpoint_latest.pt
  ✓ Loaded vae
  ✓ Loaded time_emb
  ✓ Loaded cond_encoder
  ✓ Loaded product_emb
  ✓ Loaded unet

Diffusion schedule: T=1000, β=[0.0001, 0.0200]

Checkpoint info:
  Epoch: 270
  Step: 271770
  Training loss: 0.6229


## Load Test Dataset

In [22]:
# Load test dataset
print(f"\nLoading test dataset from: {CONFIG['test_csv']}")
test_dataset = MultiProductDataset(
    pairing_csv=CONFIG['test_csv'],
    data_root=CONFIG['data_root'],
    normalize=True,
    return_census=False,
    products=CONFIG['products']
)

print(f"Test dataset size: {len(test_dataset)} samples")

# Determine evaluation size
eval_size = len(test_dataset) if CONFIG['max_eval_samples'] is None else min(CONFIG['max_eval_samples'], len(test_dataset))
print(f"Will evaluate: {eval_size} samples")


Loading test dataset from: data/paired_dataset/test_split.csv
Filtered to products: ['WorldPop']
  Before: 4,804 samples
  After:  4,804 samples
Loaded 4,804 paired samples
Product distribution:
product
WorldPop    4804
Name: count, dtype: int64
Built features lookup with 4,804 unique locations
Test dataset size: 4804 samples
Will evaluate: 4804 samples


## Generate Predictions

In [23]:
def generate_samples(models, betas, lights, settlement, product_id, device, num_samples=1, sampler='ddim', num_steps=50):
    """Generate population map samples."""
    # Create sampler
    c3ldm_sampler = C3LDMSampler(
        baseline=models['baseline'],
        vae=models['vae'],
        time_emb=models['time_emb'],
        cond_encoder=models['cond_encoder'],
        product_emb=models['product_emb'],
        unet=models['unet'],
        census_layer=None,
        betas=betas,
        device=device
    )
    
    # Prepare inputs
    if isinstance(lights, torch.Tensor):
        lights_tensor = lights.unsqueeze(0).float().to(device)
    else:
        lights_tensor = torch.from_numpy(lights).unsqueeze(0).float().to(device)
    
    if isinstance(settlement, torch.Tensor):
        settlement_tensor = settlement.unsqueeze(0).float().to(device)
    else:
        settlement_tensor = torch.from_numpy(settlement).unsqueeze(0).float().to(device)
    
    # Generate samples
    pop_maps = c3ldm_sampler.sample_population_map(
        lights_tensor,
        settlement_tensor,
        product_id,
        admin_ids=None,
        census_totals=None,
        num_samples=num_samples,
        sampler=sampler,
        num_steps=num_steps,
        eta=0.0,
        show_progress=False
    )
    
    # Convert to numpy
    samples = pop_maps.cpu().numpy()[0]  # (num_samples, 1, 256, 256)
    return samples

# Generate predictions
print(f"\nGenerating predictions using {CONFIG['sampler'].upper()}...")
print(f"Num samples per input: {CONFIG['num_samples']}")

all_predictions = []
all_targets = []
all_indices = []

with torch.no_grad():
    for idx in tqdm(range(eval_size), desc="Evaluating"):
        sample = test_dataset[idx]
        lights = sample['lights']
        settlement = sample['settlement']
        target = sample['target']
        product_id = sample['product_id']
        
        # Generate predictions
        predictions = generate_samples(
            models, betas, lights, settlement, product_id, CONFIG['device'],
            num_samples=CONFIG['num_samples'],
            sampler=CONFIG['sampler'],
            num_steps=CONFIG['num_steps']
        )
        
        # Store
        all_predictions.append(predictions)
        if isinstance(target, torch.Tensor):
            target = target.cpu().numpy()
        all_targets.append(target)
        all_indices.append(idx)

# Stack predictions and targets
all_predictions = np.concatenate(all_predictions, axis=0)  # (N*num_samples, 1, 256, 256)
all_targets = np.stack(all_targets, axis=0)  # (N, 1, 256, 256)

# Average multiple samples if num_samples > 1
if CONFIG['num_samples'] > 1:
    N = eval_size
    all_predictions = all_predictions.reshape(N, CONFIG['num_samples'], 1, 256, 256)
    all_predictions = all_predictions.mean(axis=1)  # (N, 1, 256, 256)

print(f"\nPredictions shape: {all_predictions.shape}")
print(f"Targets shape: {all_targets.shape}")


Generating predictions using DDIM...
Num samples per input: 1


Evaluating:   0%|          | 1/4804 [00:00<41:48,  1.91it/s]

pop_raw range: [0.0000, 17.1675], sum=3382.16


Evaluating:   0%|          | 2/4804 [00:00<37:20,  2.14it/s]

pop_raw range: [0.0000, 25.7054], sum=3269.79


...

Evaluating: 100%|█████████▉| 4803/4804 [33:49<00:00,  2.43it/s]

pop_raw range: [0.0000, 26.5887], sum=2235.80


Evaluating: 100%|██████████| 4804/4804 [33:49<00:00,  2.37it/s]

pop_raw range: [0.0000, 17.4563], sum=603.46



Predictions shape: (4804, 1, 256, 256)
Targets shape: (4804, 1, 256, 256)


## Compute Metrics

In [24]:
# Compute metrics
print("\nComputing metrics...")
metrics = batch_metrics(all_predictions, all_targets)

# Print metrics
print_metrics(metrics, title=f"Test Set Evaluation ({eval_size} samples)")

# Save metrics to JSON
import json
metrics_clean = {k: float(v) if not np.isnan(v) else None for k, v in metrics.items()}
with open(f"{CONFIG['output_dir']}/metrics.json", 'w') as f:
    json.dump(metrics_clean, f, indent=2)
print(f"\nSaved metrics to: {CONFIG['output_dir']}/metrics.json")


Computing metrics...

                  Test Set Evaluation (4804 samples)                  

Pixel-Level Accuracy:
----------------------------------------------------------------------
  rmse                :   2.2816
  mae                 :   0.1761
  mape                :  3416.65%
  r2                  :   0.2387

Spatial Correlation:
----------------------------------------------------------------------
  corr_pearson        :   0.5046
  corr_spearman       :   0.5352

Heavy Tail Metrics (90th percentile):
----------------------------------------------------------------------
  mae_dense           :   1.6306
  rmse_dense          :   7.1095
  mae_sparse          :   0.0145
  rmse_sparse         :   0.4103
  corr_dense          :   0.5100


Saved metrics to: eval_results/metrics.json


## Visualizations

### 1. Scatter Plot: Predicted vs Actual Population

In [ ]:
# Scatter plot: predicted vs actual
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Sample points for plotting (too many points = slow)
max_points = 10000
pred_flat = all_predictions.flatten()
target_flat = all_targets.flatten()

if len(pred_flat) > max_points:
    indices = np.random.choice(len(pred_flat), max_points, replace=False)
    pred_sample = pred_flat[indices]
    target_sample = target_flat[indices]
else:
    pred_sample = pred_flat
    target_sample = target_flat

# Plot
ax.hexbin(target_sample, pred_sample, gridsize=50, cmap='Blues', mincnt=1)
ax.plot([0, target_flat.max()], [0, target_flat.max()], 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual Population', fontsize=12)
ax.set_ylabel('Predicted Population', fontsize=12)
ax.set_title(f'Predicted vs Actual Population\nR² = {metrics["r2"]:.4f}, Pearson = {metrics["corr_pearson"]:.4f}', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/scatter_pred_vs_actual.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved scatter plot to: {CONFIG['output_dir']}/scatter_pred_vs_actual.png")

### 2. Error Distribution

In [ ]:
# Error distribution
errors = all_predictions.flatten() - all_targets.flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(errors, bins=100, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='r', linestyle='--', lw=2, label='Zero error')
axes[0].set_xlabel('Error (Predicted - Actual)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title(f'Error Distribution\nMean = {errors.mean():.4f}, Std = {errors.std():.4f}', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(errors, vert=True)
axes[1].set_ylabel('Error (Predicted - Actual)', fontsize=12)
axes[1].set_title('Error Box Plot', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/error_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved error distribution to: {CONFIG['output_dir']}/error_distribution.png")

### 3. Metrics by Population Density

In [ ]:
# Bin samples by population density
pop_sums = all_targets.sum(axis=(1, 2, 3))
bins = [0, 1, 100, 1000, np.inf]
labels = ['Zero', 'Low', 'Medium', 'High']
bin_indices = np.digitize(pop_sums, bins) - 1

# Compute metrics per bin
bin_metrics = {}
for i, label in enumerate(labels):
    mask = bin_indices == i
    if mask.sum() > 0:
        bin_preds = all_predictions[mask]
        bin_targets = all_targets[mask]
        
        bin_rmse = np.sqrt(np.mean((bin_preds - bin_targets) ** 2))
        bin_mae = np.mean(np.abs(bin_preds - bin_targets))
        
        # Correlation (flatten for correlation)
        from scipy.stats import pearsonr
        if bin_preds.size > 1:
            bin_corr, _ = pearsonr(bin_preds.flatten(), bin_targets.flatten())
        else:
            bin_corr = np.nan
        
        bin_metrics[label] = {
            'count': mask.sum(),
            'rmse': bin_rmse,
            'mae': bin_mae,
            'corr': bin_corr
        }

# Plot metrics by bin
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bin_names = list(bin_metrics.keys())
rmse_values = [bin_metrics[k]['rmse'] for k in bin_names]
mae_values = [bin_metrics[k]['mae'] for k in bin_names]
corr_values = [bin_metrics[k]['corr'] for k in bin_names]

# RMSE
axes[0].bar(bin_names, rmse_values, color='skyblue', edgecolor='black')
axes[0].set_ylabel('RMSE', fontsize=12)
axes[0].set_title('RMSE by Population Density', fontsize=14)
axes[0].grid(True, alpha=0.3, axis='y')

# MAE
axes[1].bar(bin_names, mae_values, color='lightcoral', edgecolor='black')
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].set_title('MAE by Population Density', fontsize=14)
axes[1].grid(True, alpha=0.3, axis='y')

# Correlation
axes[2].bar(bin_names, corr_values, color='lightgreen', edgecolor='black')
axes[2].set_ylabel('Pearson Correlation', fontsize=12)
axes[2].set_title('Correlation by Population Density', fontsize=14)
axes[2].set_ylim([0, 1])
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/metrics_by_density.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMetrics by population density:")
for label, values in bin_metrics.items():
    print(f"  {label:8s}: {values['count']:5d} samples, RMSE={values['rmse']:.4f}, MAE={values['mae']:.4f}, Corr={values['corr']:.4f}")

print(f"\nSaved metrics by density to: {CONFIG['output_dir']}/metrics_by_density.png")

### 4. Visual Comparison: Predictions vs Ground Truth

In [ ]:
# Visualize random samples
num_viz = min(CONFIG['num_visualize'], eval_size)
viz_indices = np.random.choice(eval_size, num_viz, replace=False)

print(f"\nVisualizing {num_viz} random samples...")

for viz_idx in viz_indices:
    # Get data
    sample = test_dataset[viz_idx]
    lights = sample['lights'].cpu().numpy()[0] if isinstance(sample['lights'], torch.Tensor) else sample['lights'][0]
    settlement = sample['settlement'].cpu().numpy()[0] if isinstance(sample['settlement'], torch.Tensor) else sample['settlement'][0]
    
    pred = all_predictions[viz_idx, 0]
    target = all_targets[viz_idx, 0]
    
    # Compute baseline for comparison
    with torch.no_grad():
        lights_tensor = torch.from_numpy(lights).unsqueeze(0).unsqueeze(0).float().to(CONFIG['device'])
        settlement_tensor = torch.from_numpy(settlement).unsqueeze(0).unsqueeze(0).float().to(CONFIG['device'])
        baseline = models['baseline'](lights_tensor, settlement_tensor).cpu().numpy()[0, 0]
    
    # Plot
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Row 1: Inputs
    im0 = axes[0, 0].imshow(lights, cmap='viridis')
    axes[0, 0].set_title('VIIRS Nighttime Lights', fontsize=12)
    axes[0, 0].axis('off')
    plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)
    
    im1 = axes[0, 1].imshow(settlement, cmap='viridis')
    axes[0, 1].set_title('WSF Settlement', fontsize=12)
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)
    
    im2 = axes[0, 2].imshow(baseline, cmap='viridis')
    axes[0, 2].set_title(f'Baseline (sum={baseline.sum():.0f})', fontsize=12)
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)
    
    # Row 2: Ground truth, Prediction, Error
    vmax = max(target.max(), pred.max())
    
    im3 = axes[1, 0].imshow(target, cmap='viridis', vmin=0, vmax=vmax)
    axes[1, 0].set_title(f'Ground Truth (sum={target.sum():.0f})', fontsize=12)
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046)
    
    im4 = axes[1, 1].imshow(pred, cmap='viridis', vmin=0, vmax=vmax)
    axes[1, 1].set_title(f'Prediction (sum={pred.sum():.0f})', fontsize=12)
    axes[1, 1].axis('off')
    plt.colorbar(im4, ax=axes[1, 1], fraction=0.046)
    
    error = pred - target
    im5 = axes[1, 2].imshow(error, cmap='RdBu_r', vmin=-vmax*0.5, vmax=vmax*0.5)
    axes[1, 2].set_title(f'Error (MAE={np.abs(error).mean():.2f})', fontsize=12)
    axes[1, 2].axis('off')
    plt.colorbar(im5, ax=axes[1, 2], fraction=0.046)
    
    # Compute correlation
    from scipy.stats import pearsonr
    corr, _ = pearsonr(pred.flatten(), target.flatten())
    
    fig.suptitle(f'Sample {viz_idx} | Correlation: {corr:.4f}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    # Save
    plt.savefig(f"{CONFIG['output_dir']}/sample_{viz_idx:04d}.png", dpi=150, bbox_inches='tight')
    plt.show()

print(f"\nSaved visualizations to: {CONFIG['output_dir']}/sample_*.png")

## Summary Report

In [ ]:
# Generate summary report
summary_path = f"{CONFIG['output_dir']}/summary.txt"

with open(summary_path, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("C3-LDM Model Evaluation Summary\n")
    f.write("=" * 70 + "\n\n")
    
    f.write("Configuration:\n")
    f.write("-" * 70 + "\n")
    f.write(f"Checkpoint: {CONFIG['checkpoint_path']}\n")
    f.write(f"Test dataset: {CONFIG['test_csv']}\n")
    f.write(f"Products: {CONFIG['products']}\n")
    f.write(f"Sampler: {CONFIG['sampler'].upper()} ({CONFIG['num_steps']} steps)\n")
    f.write(f"Samples per input: {CONFIG['num_samples']}\n")
    f.write(f"Evaluated samples: {eval_size}\n\n")
    
    f.write("Overall Metrics:\n")
    f.write("-" * 70 + "\n")
    for key, value in metrics.items():
        if np.isnan(value):
            f.write(f"  {key:25s}: NaN\n")
        elif 'rel_error' in key or 'mape' in key:
            f.write(f"  {key:25s}: {value:.2f}%\n")
        else:
            f.write(f"  {key:25s}: {value:.4f}\n")
    
    f.write("\nMetrics by Population Density:\n")
    f.write("-" * 70 + "\n")
    for label, values in bin_metrics.items():
        f.write(f"\n{label} ({values['count']} samples):\n")
        f.write(f"  RMSE: {values['rmse']:.4f}\n")
        f.write(f"  MAE: {values['mae']:.4f}\n")
        f.write(f"  Correlation: {values['corr']:.4f}\n")
    
    f.write("\n" + "=" * 70 + "\n")
    f.write("Evaluation completed successfully!\n")
    f.write("=" * 70 + "\n")

print(f"\nSaved summary report to: {summary_path}")
print("\n" + "=" * 70)
print("Evaluation completed!")
print("=" * 70)

## Optional: Save Predictions

In [ ]:
if CONFIG['save_predictions']:
    pred_dir = Path(CONFIG['output_dir']) / 'predictions'
    pred_dir.mkdir(exist_ok=True)
    
    print(f"\nSaving all predictions to: {pred_dir}")
    
    for idx in tqdm(range(eval_size), desc="Saving predictions"):
        np.save(pred_dir / f'pred_{idx:06d}.npy', all_predictions[idx])
        np.save(pred_dir / f'target_{idx:06d}.npy', all_targets[idx])
    
    print(f"Saved {eval_size} predictions and targets")
else:
    print("\nSkipping prediction saving (save_predictions=False)")